# Gaze CNN Training

Train a 4-layer CNN on MPIIGaze for 5-class gaze estimation.

Run this notebook on Google Colab (Runtime → Change runtime type → T4 GPU).

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision numpy opencv-python matplotlib scikit-learn tqdm

## 2. Download MPIIGaze

In [ ]:
import os, urllib.request, zipfile

# Download MPIIGaze (alternatively: upload your copy)
# This is a placeholder - MPIIGaze requires manual download from MPI website
# https://www.mpi-inf.mpg.de/departments/computer-vision-and-machine-learning/research/gaze-based-human-computer-interaction/mpiigaze

print("Download MPIIGaze from the official link and extract to data/gaze/mpiigaze/")
print("Or upload your copy to Colab.")

## 3. Imports + Dataset

In [ ]:
import sys
sys.path.append('/content/ai-gimbal-camera')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import json, os, time

# Copy src/training/* into this notebook or mount from GitHub
print("Ready")

## 4. Model Definition

In [ ]:
class GazeCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256, 128),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x).view(x.size(0), -1)
        return self.classifier(x)

model = GazeCNN()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Data Loading (adjust path to your MPIIGaze location)

In [ ]:
DATA_DIR = 'data/gaze/mpiigaze'  # Change this

# Simplified loader for this notebook
def load_mpiigaze(data_dir, subject_ids):
    images, labels = [], []
    for sid in subject_ids:
        subj_dir = os.path.join(data_dir, f'p{sid:02d}')
        label_file = os.path.join(subj_dir, 'label.txt')
        if not os.path.exists(label_file):
            continue
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7: continue
                img_path = os.path.join(subj_dir, parts[0])
                gaze = np.array([float(parts[1]), float(parts[2]), float(parts[3])])
                label = 0 if np.linalg.norm(gaze[:2]) < 0.2 else (
                    1 if gaze[0] < -0.2 else (
                        2 if gaze[0] > 0.2 else (
                            3 if gaze[1] > 0.2 else 4
                        )
                    )
                )
                import cv2
                img = cv2.imread(img_path)
                if img is None: continue
                img = cv2.resize(img, (128, 128)).astype(np.float32) / 255.0
                img = np.transpose(img, (2, 0, 1))
                images.append(img)
                labels.append(label)
    return np.array(images), np.array(labels)

print("Loader defined")

## 6. Training (Leave-One-Person-Out)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

all_subjects = list(range(15))
subject_accs = []

for test_subject in all_subjects:
    train_subjects = [s for s in all_subjects if s != test_subject]
    
    X_train, y_train = load_mpiigaze(DATA_DIR, train_subjects)
    X_test, y_test = load_mpiigaze(DATA_DIR, [test_subject])
    
    if len(X_test) == 0:
        print(f'Skipping subject {test_subject}: no data')
        continue
    
    train_dataset = [(torch.FloatTensor(x), y) for x, y in zip(X_train, y_train)]
    test_dataset = [(torch.FloatTensor(x), y) for x, y in zip(X_test, y_test)]
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64)
    
    model = GazeCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    for epoch in range(30):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs).argmax(dim=1)
            correct += (preds == labels).sum().item()
    
    acc = correct / len(test_dataset)
    subject_accs.append(acc)
    print(f'Subject {test_subject}: {acc:.4f}')

print(f'\nMean accuracy: {np.mean(subject_accs):.4f} +/- {np.std(subject_accs):.4f}')

## 7. Train Final Model on All Data

In [ ]:
X_all, y_all = load_mpiigaze(DATA_DIR, all_subjects)
all_dataset = [(torch.FloatTensor(x), y) for x, y in zip(X_all, y_all)]
all_loader = DataLoader(all_dataset, batch_size=64, shuffle=True)

final_model = GazeCNN().to(device)
optimizer = optim.Adam(final_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    final_model.train()
    loss_sum = 0
    for inputs, labels in all_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(inputs), labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/50: Loss={loss_sum/len(all_loader):.4f}')

# Save
torch.save(final_model.state_dict(), 'gaze_cnn.pth')
print('Model saved to gaze_cnn.pth')

# Download to your machine
from google.colab import files
files.download('gaze_cnn.pth')